# Phase 9 — Backend API and storage

Self-contained FastAPI service, typed contracts, SQLAlchemy schema, deterministic predictions/simulations, structured errors and logs, service metrics, and isolated SQLite integration tests. Production uses PostgreSQL through `MATCHCAST_DATABASE_URL`; tests use isolated SQLite. Every prediction and simulation stores its model version and input metadata.

In [1]:
from __future__ import annotations
import json, logging, math, os, time, uuid
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, PlainTextResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, model_validator
from sqlalchemy import Date, DateTime, Float, ForeignKey, Integer, JSON, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy.pool import StaticPool

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_VERSION = 'elo-outcome-1.0.0'
MAX_RUNS = int(os.getenv('MATCHCAST_MAX_SIMULATIONS', '10000'))
logging.basicConfig(level=os.getenv('MATCHCAST_LOG_LEVEL', 'INFO'), format='%(message)s')
logger = logging.getLogger('matchcast')
METRICS = Counter()

class Base(DeclarativeBase): pass
class TeamRow(Base):
    __tablename__='teams'; id: Mapped[int]=mapped_column(primary_key=True); name: Mapped[str]=mapped_column(String(120),unique=True,index=True); rating: Mapped[float]=mapped_column(Float)
class MatchRow(Base):
    __tablename__='matches'; id: Mapped[int]=mapped_column(primary_key=True); played_on: Mapped[date]=mapped_column(Date); home_team_id: Mapped[int]=mapped_column(ForeignKey('teams.id')); away_team_id: Mapped[int]=mapped_column(ForeignKey('teams.id')); home_score: Mapped[int]=mapped_column(Integer); away_score: Mapped[int]=mapped_column(Integer)
class ModelRow(Base):
    __tablename__='model_versions'; id: Mapped[int]=mapped_column(primary_key=True); version: Mapped[str]=mapped_column(String(80),unique=True); model_type: Mapped[str]=mapped_column(String(80)); metrics: Mapped[dict]=mapped_column(JSON); created_at: Mapped[datetime]=mapped_column(DateTime(timezone=True),default=lambda:datetime.now(timezone.utc))
class PredictionRow(Base):
    __tablename__='predictions'; id: Mapped[str]=mapped_column(String(36),primary_key=True); model_version: Mapped[str]=mapped_column(String(80)); inputs: Mapped[dict]=mapped_column(JSON); probabilities: Mapped[dict]=mapped_column(JSON); created_at: Mapped[datetime]=mapped_column(DateTime(timezone=True),default=lambda:datetime.now(timezone.utc))
class SimulationRow(Base):
    __tablename__='simulations'; id: Mapped[str]=mapped_column(String(36),primary_key=True); model_version: Mapped[str]=mapped_column(String(80)); inputs: Mapped[dict]=mapped_column(JSON); seed: Mapped[int]=mapped_column(Integer); runs: Mapped[int]=mapped_column(Integer); status: Mapped[str]=mapped_column(String(20)); created_at: Mapped[datetime]=mapped_column(DateTime(timezone=True),default=lambda:datetime.now(timezone.utc))
class SimulationResultRow(Base):
    __tablename__='simulation_results'; id: Mapped[int]=mapped_column(primary_key=True); simulation_id: Mapped[str]=mapped_column(ForeignKey('simulations.id'),index=True); team: Mapped[str]=mapped_column(String(120)); finish_probabilities: Mapped[dict]=mapped_column(JSON); qualification_probability: Mapped[float]=mapped_column(Float)

class OutcomeProbabilities(BaseModel):
    home_win: float=Field(ge=0,le=1); draw: float=Field(ge=0,le=1); away_win: float=Field(ge=0,le=1)
    @model_validator(mode='after')
    def normalized(self):
        if not math.isclose(self.home_win+self.draw+self.away_win,1.0,abs_tol=1e-8): raise ValueError('probabilities must sum to one')
        return self
class PredictRequest(BaseModel): home_team:str; away_team:str; neutral:bool=False
class PredictResponse(BaseModel): prediction_id:str; model_version:str; probabilities:OutcomeProbabilities
class SimulationRequest(BaseModel): teams:list[str]=Field(min_length=2,max_length=16); runs:int=Field(default=1000,ge=1); seed:int=20260705; format:Literal['round_robin']='round_robin'; qualifiers:int=Field(default=2,ge=1)
    
def build_engine(url: str|None=None):
    url=url or os.getenv('MATCHCAST_DATABASE_URL','sqlite+pysqlite:///./matchcast.db')
    kw={'connect_args':{'check_same_thread':False}} if url.startswith('sqlite') else {}
    if url=='sqlite+pysqlite:///:memory:': kw['poolclass']=StaticPool
    return create_engine(url,**kw)

def latest_ratings():
    d=pd.read_csv(ROOT/'data/processed/matches_with_elo.csv',usecols=['date','home_team','away_team','home_elo','away_elo'])
    a=d[['date','home_team','home_elo']].rename(columns={'home_team':'team','home_elo':'rating'}); b=d[['date','away_team','away_elo']].rename(columns={'away_team':'team','away_elo':'rating'})
    return pd.concat([a,b]).sort_values('date').groupby('team').tail(1).set_index('team')['rating'].to_dict()

def probabilities(home:str,away:str,neutral:bool,ratings:dict[str,float]):
    advantage=0 if neutral else 100; raw=1/(1+10**(-((ratings[home]-ratings[away]+advantage)/400))); draw=.25
    return OutcomeProbabilities(home_win=(1-draw)*raw,draw=draw,away_win=(1-draw)*(1-raw))

def create_app(engine=None,ratings=None):
    engine=engine or build_engine(); ratings=ratings or latest_ratings(); Base.metadata.create_all(engine)
    with Session(engine) as db:
        if not db.scalar(select(ModelRow).where(ModelRow.version==MODEL_VERSION)): db.add(ModelRow(version=MODEL_VERSION,model_type='Elo outcome baseline',metrics={'world_cup_log_loss':0.993}))
        existing=set(db.scalars(select(TeamRow.name))); db.add_all([TeamRow(name=t,rating=r) for t,r in ratings.items() if t not in existing]); db.commit()
    app=FastAPI(title='MatchCast API',version='1.0.0')
    @app.middleware('http')
    async def observe(request:Request,call_next):
        start=time.perf_counter(); response=await call_next(request); elapsed=time.perf_counter()-start; METRICS[f'requests_total|{request.method}|{request.url.path}|{response.status_code}']+=1; METRICS['latency_ms_total']+=round(elapsed*1000); logger.info(json.dumps({'event':'request','method':request.method,'path':request.url.path,'status':response.status_code,'duration_ms':round(elapsed*1000,2)})); return response
    @app.exception_handler(HTTPException)
    async def http_error(request,exc): return JSONResponse(status_code=exc.status_code,content={'error':{'code':'request_error','message':str(exc.detail)}})
    @app.get('/health')
    def health(): return {'status':'ok','model_version':MODEL_VERSION}
    @app.get('/teams')
    def teams(): return {'teams':sorted(ratings)}
    @app.post('/predict-match',response_model=PredictResponse)
    def predict(body:PredictRequest):
        if body.home_team==body.away_team: raise HTTPException(422,'home and away teams must differ')
        unknown=[t for t in [body.home_team,body.away_team] if t not in ratings]
        if unknown: raise HTTPException(404,f'unknown team: {unknown[0]}')
        p=probabilities(body.home_team,body.away_team,body.neutral,ratings); pid=str(uuid.uuid4())
        with Session(engine) as db: db.add(PredictionRow(id=pid,model_version=MODEL_VERSION,inputs=body.model_dump(),probabilities=p.model_dump())); db.commit()
        return PredictResponse(prediction_id=pid,model_version=MODEL_VERSION,probabilities=p)
    @app.post('/simulate-tournament',status_code=201)
    def simulate(body:SimulationRequest):
        if len(body.teams)!=len(set(body.teams)): raise HTTPException(422,'teams must be unique')
        if body.runs>MAX_RUNS: raise HTTPException(422,f'runs must not exceed {MAX_RUNS}')
        if body.qualifiers>=len(body.teams): raise HTTPException(422,'qualifiers must be less than team count')
        unknown=[t for t in body.teams if t not in ratings]
        if unknown: raise HTTPException(404,f'unknown team: {unknown[0]}')
        rng=np.random.default_rng(body.seed); counts={t:np.zeros(len(body.teams),int) for t in body.teams}
        for _ in range(body.runs):
            table={t:[0,0,0] for t in body.teams}
            for i,h in enumerate(body.teams):
                for a in body.teams[i+1:]:
                    p=probabilities(h,a,True,ratings); outcome=rng.choice(3,p=[p.home_win,p.draw,p.away_win]); hg,ag=((1,0) if outcome==0 else (0,0) if outcome==1 else (0,1)); table[h][1]+=hg-ag;table[a][1]+=ag-hg;table[h][2]+=hg;table[a][2]+=ag; table[h][0]+=3 if outcome==0 else 1 if outcome==1 else 0;table[a][0]+=3 if outcome==2 else 1 if outcome==1 else 0
            order=sorted(body.teams,key=lambda t:(-table[t][0],-table[t][1],-table[t][2],t)); [counts[t].__setitem__(pos,counts[t][pos]+1) for pos,t in enumerate(order)]
        sid=str(uuid.uuid4()); rows=[]
        with Session(engine) as db:
            db.add(SimulationRow(id=sid,model_version=MODEL_VERSION,inputs=body.model_dump(),seed=body.seed,runs=body.runs,status='completed'))
            for t in body.teams:
                fp={str(i+1):float(v/body.runs) for i,v in enumerate(counts[t])}; q=sum(fp[str(i+1)] for i in range(body.qualifiers)); db.add(SimulationResultRow(simulation_id=sid,team=t,finish_probabilities=fp,qualification_probability=q)); rows.append({'team':t,'finish_probabilities':fp,'qualification_probability':q})
            db.commit()
        return {'simulation_id':sid,'status':'completed','model_version':MODEL_VERSION,'results':rows}
    @app.get('/simulation/{simulation_id}')
    def get_simulation(simulation_id:str):
        with Session(engine) as db:
            sim=db.get(SimulationRow,simulation_id)
            if not sim: raise HTTPException(404,'simulation not found')
            rows=db.scalars(select(SimulationResultRow).where(SimulationResultRow.simulation_id==simulation_id)).all()
            return {'simulation_id':sim.id,'status':sim.status,'model_version':sim.model_version,'results':[{'team':r.team,'finish_probabilities':r.finish_probabilities,'qualification_probability':r.qualification_probability} for r in rows]}
    @app.get('/models/leaderboard')
    def leaderboard(): return {'models':[{'rank':1,'name':'logistic-regression','log_loss':.982},{'rank':2,'name':'elo','log_loss':.993},{'rank':3,'name':'poisson','log_loss':.998}]}
    @app.get('/metrics',response_class=PlainTextResponse)
    def metrics(): return '\n'.join(f'matchcast_{k.replace("|","_")} {v}' for k,v in METRICS.items())+'\n'
    return app

engine=build_engine('sqlite+pysqlite:///:memory:'); test_ratings={'Argentina':2020.,'England':1940.,'France':1910.,'Spain':1890.}; app=create_app(engine,test_ratings); client=TestClient(app)
assert client.get('/health').status_code==200 and client.get('/teams').json()['teams']==sorted(test_ratings)
pred=client.post('/predict-match',json={'home_team':'Argentina','away_team':'England','neutral':True}); assert pred.status_code==200 and math.isclose(sum(pred.json()['probabilities'].values()),1)
assert client.post('/predict-match',json={'home_team':'X','away_team':'England'}).status_code==404
assert client.post('/predict-match',json={'home_team':'England','away_team':'England'}).status_code==422
payload={'teams':list(test_ratings),'runs':100,'seed':7,'qualifiers':2}; sim=client.post('/simulate-tournament',json=payload); assert sim.status_code==201
got=client.get('/simulation/'+sim.json()['simulation_id']); assert got.status_code==200 and sum(r['qualification_probability'] for r in got.json()['results'])==2
sim2=client.post('/simulate-tournament',json=payload); assert sim.json()['results']==sim2.json()['results']
assert client.get('/simulation/missing').status_code==404 and client.get('/models/leaderboard').status_code==200 and client.get('/metrics').status_code==200
with Session(engine) as db: assert db.scalar(select(PredictionRow)).model_version==MODEL_VERSION and db.scalar(select(SimulationRow)).inputs['seed']==7
assert set(Base.metadata.tables)=={'teams','matches','model_versions','predictions','simulations','simulation_results'}
print('All Phase 9 API, validation, persistence, repeatability, and isolation checks passed.')
if os.getenv('MATCHCAST_RUNTIME')=='1': app=create_app()

G:\Coding\nations-ai\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{"event": "request", "method": "GET", "path": "/health", "status": 200, "duration_ms": 2.78}


HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


{"event": "request", "method": "GET", "path": "/teams", "status": 200, "duration_ms": 1.95}


HTTP Request: GET http://testserver/teams "HTTP/1.1 200 OK"


...

{"event": "request", "method": "GET", "path": "/models/leaderboard", "status": 200, "duration_ms": 2.03}


HTTP Request: GET http://testserver/models/leaderboard "HTTP/1.1 200 OK"


{"event": "request", "method": "GET", "path": "/metrics", "status": 200, "duration_ms": 1.77}


HTTP Request: GET http://testserver/metrics "HTTP/1.1 200 OK"


All Phase 9 API, validation, persistence, repeatability, and isolation checks passed.
